# Week 01 · Fashion-MNIST Baseline

**Quarter:** Q1  
**Date started:** _2026-10-14 
**Status:** Completed

## Objective

_Describe the assignment goal and what the baseline should establish._

## Reproducibility checklist

- [ ] Seed recorded and set for `random`, `numpy`, and `torch`
- [ ] Train/validation/test indices saved to `Q1/data/split_indices/`
- [ ] Run configuration saved to `Q1/configs/`
- [ ] Raw metrics saved to `Q1/results/raw/`
- [ ] Figures saved to `Q1/figures/`
- [ ] Any outside or AI help logged in `Q1/assistance_log.md`

## 0. Setup

Paths, device selection (MPS with CPU fallback), and environment versions.

In [3]:
%pip install torch torchvision torchaudio


Note: you may need to restart the kernel to use updated packages.


In [4]:


import platform
from pathlib import Path

import torch

# Paths relative to this notebook (Q1/notebooks/)
Q1_DIR = Path.cwd().parent
DATA_DIR = Q1_DIR / "data"
SPLIT_DIR = DATA_DIR / "split_indices"
CONFIG_DIR = Q1_DIR / "configs"
RESULTS_DIR = Q1_DIR / "results" / "raw"
FIGURES_DIR = Q1_DIR / "figures"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print(f"Python {platform.python_version()} | torch {torch.__version__} | device: {device}")

Python 3.14.0 | torch 2.14.0 | device: mps


In [5]:
## 1. Data

# TODO
#Torchvision is where I am going to be importing the dataset from.
from torchvision import datasets, transforms

transform=transforms.ToTensor()
# This is the type of model that I will be using for the base model of traning. It has iamges.
full_train_dataset = datasets.FashionMNIST(
root="../data",
train=True,
download=True,#This basically means if the dataset is not in the interent then it willl donwload from the internet.
transform=transform
)


print(len(full_train_dataset))  # Should print 60000
# This is value is basically printing all the data inside this imported dataset that  I will be using for this base model of traning. 


60000


In [6]:
image, label = full_train_dataset[0]
print("Image Shape:", image.shape)  # Should print torch.Size([1, 28, 28])
print("Label:", label)  # Should print the label for the image, which correlations to the annotation of the image. For example, if the label is 0 then it means that the image is a t-shirt/top. If the label is 1 then it means that the image is a trouser. If the label is 2 then it means that the image is a pullover. If the label is 3 then it means that the image is a dress. If the label is 4 then it means that the image is a coat. If the label is 5 then it means that the image is a sandal. If the label is 6 then it means that the image is a shirt. If the label is 7 then it means that the image is a sneaker. If the label is 8 then it means that the image is a bag. If the label is 9 then it means that the image is an ankle boot.
print("Minimum pixel value:", image.min().item())#This prints the pixel value for the image
print("Maximum pixel value:", image.max().item())#This prings the maxiumum pixel value for the image when it enlarges

Image Shape: torch.Size([1, 28, 28])
Label: 9
Minimum pixel value: 0.0
Maximum pixel value: 1.0


In [ ]:
## 2. Train / validation / test splits

import torch
from torch.utils.data import Subset
SEED=42
#Randomly shuffles the data that is going to be used for the traning of the model and it will shuffle the data the same way
generator = torch.Generator().manual_seed(SEED)
#This is basically shuffling the data that is going to be used for the traning of the model and it will shuffle the data the same way
indices=torch.randperm(len(full_train_dataset), generator=generator)
#This is basically taking the first 5000 images from the shuffled data and it will be used for the validation of the model
val_indices=indices[:5000]
#This resvewres some of the daat set for traning the model on
train_indices = indices[5000:15000]
#This prints the number of traning examples the len prints the number of the indicies that are in it. 
print("Training examples:", len(train_indices))
print("Validation examples:", len(val_indices))#This prints the number of validation examples that will be used

Training examples: 10000
Validation examples: 5000


In [ ]:
# This makes sure that there is no ovnwerlap between the traing examples and the validation examples.
overlap = set(train_indices.tolist()).intersection(set(val_indices.tolist()))
print("Number of overlapping examples:", len(overlap))


Number of overlapping examples: 0


In [9]:
train_dataset = Subset(full_train_dataset, train_indices)
val_dataset = Subset(full_train_dataset, val_indices)

# The two lines of code on the top turn the train and vaalidation into actual daatsets that the model can be used to program on becausse without it is just a list of tensors in the dataset.
print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

Train dataset: 10000
Validation dataset: 5000


In [10]:
import numpy as np
np.save("../data/split_indices/train_indices.npy", train_indices.numpy())
np.save("../data/split_indices/val_indices.npy", val_indices.numpy())

#This saves the split indicies which basically it means it saves the order of the indicies that it was shuffled in



In [11]:
from torch.utils.data import DataLoader
BACH_SIZE=32
train_loader = DataLoader(train_dataset, batch_size=BACH_SIZE, shuffle=True)
test_loader= DataLoader(val_dataset, batch_size=BACH_SIZE, shuffle=False)


In [12]:
images, labels = next(iter(train_loader))#This basically iterates through the loader and shows each of the image shape and label

print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)
print("Labels:", labels)

Batch image shape: torch.Size([32, 1, 28, 28])
Batch label shape: torch.Size([32])
Labels: tensor([2, 6, 4, 4, 9, 3, 1, 7, 6, 7, 7, 2, 3, 8, 2, 9, 0, 0, 4, 5, 0, 5, 5, 0,
        2, 8, 9, 7, 6, 8, 8, 2])


## 3. Model

_TODO_

In [ ]:
import torch.nn as nn# Import PyTorch's neural-network module.

class LinearClassifier(nn.Module):
    def __init__(self):
        super().__init__()
# nn.Flatten() converts each image into one long vector.
        self.flatten = nn.Flatten()

        self.linear = nn.Linear(28 * 28, 10)


    def forward(self, x):
        # Flatten each 28 x 28 image into a vector containing 784 values.
        x = self.flatten(x)
        # Return the model's 10 output scores for each image.
        x = self.linear(x)
        return x
# selected earlier, such as Apple's MPS accelerator or the CPU.
model = LinearClassifier().to(device)

print(model)

LinearClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear): Linear(in_features=784, out_features=10, bias=True)
)


## 4. Training

_TODO_

In [14]:
import torch.optim as optim
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)


In [ ]:
# Number of times the model will go through the entire training dataset.
# One total EPCH means a 10,000 examples per set.
EPOCHS = 5
for epoch in range(EPOCHS):
    # This tells PyTorch that the model is currently being trained.
    model.train()
    total_loss=0.0
    for images, labels in train_loader:
        # Move the batch of images to the same device as the model.
        # On my Mac this is MPS; otherwise it could be CPU.
        images = images.to(device)
        labels = labels.to(device)
         # Clear gradients left over from the previous batch.
        optimizer.zero_grad()
          # Send the images through the model.
        outputs = model(images)
        # Use the gradients to update the model's weights and biases.
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss = total_loss+ loss.item()
    average_loss = total_loss / len(train_loader)
    print(
        f"Epoch {epoch+1}/{EPOCHS}, "
        f"Loss: {average_loss:.4f}"
    )

Epoch 1/5, Loss: 1.2879
Epoch 2/5, Loss: 0.8609
Epoch 3/5, Loss: 0.7565
Epoch 4/5, Loss: 0.6990
Epoch 5/5, Loss: 0.6615


## 5. Evaluation

_TODO_

In [17]:
model.eval()
correct = 0
total = 0
val_loss = 0.0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        val_loss += loss.item()
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
average_val_loss = val_loss / len(test_loader)
accuracy = correct / total
print(f"Validation Loss: {average_val_loss:.4f}")
print(f"Validation Accuracy: {accuracy * 100:.2f}%")

Validation Loss: 0.6651
Validation Accuracy: 78.42%


## 6. Results and figures

_TODO_

## Results

The linear Fashion-MNIST classifier was trained for 5 epochs on a fixed
10,000-example training subset.

Training loss decreased from 1.2879 to 0.6615.

On the fixed 5,000-example validation set, the model achieved:

- Validation loss: 0.6651
- Validation accuracy: 78.42%

The similar training and validation losses suggest that the model did not
show substantial overfitting during this baseline run.

## 7. Reflection

_What worked, what didn't, and what to try next._

So as for the relction I learned quite a bit about the fashion mnist dataset and how to use it to train a model. I learned how to import the dataset, split it into training and validation sets, and create data loaders for each set. I also learned how to define a simple linear classifier model using PyTorch, and how to train and evaluate the model using the training and validation sets. Overall, this was a great introduction to working with the fashion mnist dataset and training models in PyTorch. Furthermore, I learned how to effectivley manipulate a tesnor and how to prevent a model from overfitting.